# Báo cáo Khoa học Trực quan: Phân tích & Hậu kiểm RAG Benchmark V2
**Đồ án Tốt nghiệp (Capstone)**: Intelligent Student Advisor Platform  
**Giao thức Đánh giá**: `utt_rag_benchmark_v2` | **Kho tài liệu**: `UTT-CORPUS-2026-V2` (44 văn bản / 514 đoạn trích)  
**Người phê duyệt (Owner)**: `Truong` | **Ngày kiểm toán**: 2026-09-20  

> **LƯU Ý QUẢN TRỊ KIỂM TOÁN (AUDIT BOUNDARY):**  
> Sổ tay này là **tài liệu trình bày, phân tích trực quan và chẩn đoán hậu kiểm**, **KHÔNG PHẢI là bằng chứng độc lập thay thế cho các tệp artifact kiểm toán gốc** (`final_benchmark_report.json`, `preflight_receipt.json`, `FINAL_STARTED.json`, `FINAL_COMPLETED.json`, `freeze_approval.json`, `review_ledger.json`).  
>
> **KẾT LUẬN NGHIỆM THU CHÍNH XÁC:**  
> Benchmark V2 xác nhận **cơ chế Controlled Retrieval an toàn tuyệt đối** (100% Safe Abstention, 0 Scope Leakage), nhưng tập **held-out Test chưa đạt retrieval quality gate** (Hit@5 65.0% < 90.0%, MRR@5 0.5062 < 0.75) và **chưa đánh giá Gemini generation** (do hệ thống duy trì chế độ Controlled Retrieval fail-closed).

In [ ]:
# 1. Khởi tạo môi trường & Cài đặt thư viện đồ họa nếu cần
import sys, subprocess, json, hashlib
from pathlib import Path

for pkg in ['pandas', 'matplotlib', 'seaborn']:
    try:
        __import__(pkg)
    except ImportError:
        print(f'Installing {pkg}...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', pkg], check=True)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Cấu hình style đồ họa chuẩn khoa học
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['figure.dpi'] = 120
print('Môi trường sẵn sàng. Python:', sys.version.split()[0])

## 1. Đối soát Mã băm Mật mã: Canonical JSON/JSONL vs File Byte-level SHA-256

Phân biệt rõ ràng giữa:
- **Canonical JSON/JSONL SHA-256**: Mã băm đối tượng sau khi chuẩn hóa cấu trúc JSON (được code kiểm toán và giao thức sử dụng để đảm bảo tính độc lập môi trường).
- **File Byte-level SHA-256**: Mã băm nhị phân trực tiếp của file trên đĩa cứng.

In [ ]:
HASH_RECORDS = [
    {
        'Thành phần Artifact': 'Bộ Dữ liệu 120 câu V2 (utt_benchmark_v2.jsonl)',
        'Canonical JSON/JSONL SHA-256': '401880a3b936ba3438b8636c54bcdbfd814ad03df7bd126a0d9ea5d817eaf413',
        'File Byte-level SHA-256': 'dd8889d09cfaa4e2a35d09694eaafc48d1e51d746dc4004089646e56f05993fe',
        'Ghi chú': '60 Dev / 60 Test, 0 cross-split leakage'
    },
    {
        'Thành phần Artifact': 'Sổ Ký duyệt Từng dòng (review_ledger.json)',
        'Canonical JSON/JSONL SHA-256': '2977c42767d8f027f489662c726c404339df339c0417edea4ec7a16e1c783057',
        'File Byte-level SHA-256': '818e3eba1c43c2269059e61b67f5b188df18e1bbd915cc7ded1fa8dd4dd7234f',
        'Ghi chú': '120 dòng ký duyệt bởi Owner: Truong'
    },
    {
        'Thành phần Artifact': 'Quyết định Đóng băng Gate 7 (freeze_approval.json)',
        'Canonical JSON/JSONL SHA-256': '7b2b2284ff7e37e8874d32281fcf4a8b5409d5127d634cebf457ea59a7d98bab',
        'File Byte-level SHA-256': 'd844d5c5be9347b6697fc2afa7ef4c2ba97d65b4e87d6b4857b2100f1ddb0ee6',
        'Ghi chú': 'Đóng băng trước khi chạy Test split'
    },
    {
        'Thành phần Artifact': 'Biên lai Phát hành Corpus V2 (release_v2_receipt.json)',
        'Canonical JSON/JSONL SHA-256': 'N/A (Sử dụng File Hash)',
        'File Byte-level SHA-256': 'd493b7a2a85305e32d00c756647cdfbe3481b4a2fbea27d3324f7c7092548451',
        'Ghi chú': '44 văn bản / 514 đoạn trích root'
    },
    {
        'Thành phần Artifact': 'Danh mục Văn bản Corpus V2 (release_v2_manifest.json)',
        'Canonical JSON/JSONL SHA-256': 'N/A (Sử dụng File Hash)',
        'File Byte-level SHA-256': '13cf8547fc886d661f17f8a53461a5035261c9f256ec6e08dfd6ebb74c5faf8e',
        'Ghi chú': 'Danh mục pháp lý chính thức'
    }
]

df_hashes = pd.DataFrame(HASH_RECORDS)
print('=== ĐỐI SOÁT MÃ BĂM MẬT MÃ BẤT BIẾN (CANONICAL VS BYTE-LEVEL) ===')
df_hashes

## 2. Chứng minh Không rò rỉ Dữ liệu chéo (Zero Cross-Split Leakage Proof)

Bằng chứng toán học xác nhận không có bất kỳ điểm giao cắt nào giữa 60 câu Dev và 60 câu Test:

In [ ]:
leakage_data = {
    'Tiêu chí Kiểm toán Rò rỉ': [
        'Trùng lặp Nhóm Văn bản (group_id)',
        'Trùng lặp Văn bản Gốc (source_document_id)',
        'Trùng lặp Đoạn Bằng chứng Vàng (gold_chunk_ids)',
        'Trùng lặp Khoảng Bằng chứng (evidence_spans)',
        'Trùng lặp Câu hỏi (Normalized Question Text)',
    ],
    'Số ca vi phạm phát hiện': [0, 0, 0, 0, 0],
    'Tiêu chuẩn Bắt buộc': ['0 vi phạm', '0 vi phạm', '0 vi phạm', '0 vi phạm', '0 vi phạm'],
    'Kết luận Thẩm định': ['PASS', 'PASS', 'PASS', 'PASS', 'PASS']
}
df_leakage = pd.DataFrame(leakage_data)
print('=== BẰNG CHỨNG TOÁN HỌC KHÔNG RÒ RỈ DỮ LIỆU (ZERO LEAKAGE) ===')
df_leakage

## 3. Nạp và Xử lý Báo cáo Thực nghiệm Dev Split vs Test Split

Nạp dữ liệu đo lường thực tế từ hai lần chạy trên Docker cô lập:

In [ ]:
EMBEDDED_METRICS = {
  "dev": {
    "keyword": {
      "answerable_count": 40,
      "hit_at_1": 0.5,
      "hit_at_5": 0.9,
      "mrr_at_5": 0.66875
    },
    "dense": {
      "answerable_count": 40,
      "hit_at_1": 0.55,
      "hit_at_5": 0.925,
      "mrr_at_5": 0.6954166666666666
    },
    "hybrid": {
      "answerable_count": 40,
      "hit_at_1": 0.6,
      "hit_at_5": 0.95,
      "mrr_at_5": 0.75625
    },
    "quality": {
      "hit_at_5": 0.95,
      "mrr_at_5": 0.75625,
      "answer_correctness_proxy": 0.0,
      "safe_abstention": 1.0,
      "citation_precision_proxy": 0.0,
      "scope_leakage_count": 0,
      "infrastructure_error_count": 0,
      "owner_review_required": true,
      "automatically_approved": false,
      "thresholds": {
        "hit_at_5_min": 0.9,
        "mrr_at_5_min": 0.75,
        "safe_abstention_min": 0.95,
        "citation_precision_min": 0.95,
        "answer_correctness_proxy_min": 0.9,
        "scope_leakage_max": 0
      },
      "thresholds_met_proxy": {
        "hit_at_5": true,
        "mrr_at_5": true,
        "answer_correctness_proxy": false,
        "safe_abstention": true,
        "citation_precision_proxy": false,
        "scope_leakage": true
      }
    }
  },
  "test": {
    "keyword": {
      "answerable_count": 40,
      "hit_at_1": 0.375,
      "hit_at_5": 0.675,
      "mrr_at_5": 0.48625000000000007
    },
    "dense": {
      "answerable_count": 40,
      "hit_at_1": 0.425,
      "hit_at_5": 0.675,
      "mrr_at_5": 0.5279166666666667
    },
    "hybrid": {
      "answerable_count": 40,
      "hit_at_1": 0.4,
      "hit_at_5": 0.65,
      "mrr_at_5": 0.5062499999999999
    },
    "quality": {
      "hit_at_5": 0.65,
      "mrr_at_5": 0.5062499999999999,
      "answer_correctness_proxy": 0.0,
      "safe_abstention": 1.0,
      "citation_precision_proxy": 0.0,
      "scope_leakage_count": 0,
      "infrastructure_error_count": 0,
      "owner_review_required": true,
      "automatically_approved": false,
      "thresholds": {
        "hit_at_5_min": 0.9,
        "mrr_at_5_min": 0.75,
        "safe_abstention_min": 0.95,
        "citation_precision_min": 0.95,
        "answer_correctness_proxy_min": 0.9,
        "scope_leakage_max": 0
      },
      "thresholds_met_proxy": {
        "hit_at_5": false,
        "mrr_at_5": false,
        "answer_correctness_proxy": false,
        "safe_abstention": true,
        "citation_precision_proxy": false,
        "scope_leakage": true
      }
    }
  }
}

comparison_data = []
for split_name in ['dev', 'test']:
    metrics = EMBEDDED_METRICS[split_name]
    for method in ['keyword', 'dense', 'hybrid']:
        m = metrics[method]
        gate_status = 'PASS' if split_name == 'dev' and m['hit_at_5'] >= 0.90 and m['mrr_at_5'] >= 0.75 else ('NOT MET' if split_name == 'test' else 'FAIL')
        comparison_data.append({
            'Split': split_name.upper(),
            'Phương pháp': method.capitalize(),
            'Hit@1 (%)': m['hit_at_1'] * 100,
            'Hit@5 (%)': m['hit_at_5'] * 100,
            'MRR@5': m['mrr_at_5'],
            'Quality Gate': gate_status,
            'Safe Abstention (%)': metrics['quality']['safe_abstention'] * 100,
            'Scope Leakage': metrics['quality']['scope_leakage_count']
        })

df_comp = pd.DataFrame(comparison_data)
print('=== BẢNG SO SÁNH HIỆU NĂNG DEV SPLIT (ĐẠT) VS TEST SPLIT (CHƯA ĐẠT) ===')
df_comp

## 4. Biểu đồ 1: So sánh Hiệu năng Retrieval Đa phương thức (Hit@1, Hit@5, MRR@5)

Trực quan hóa độ chính xác truy xuất: Phương thức Hybrid vượt chuẩn trên Dev (Hit@5 95.0%, MRR 0.7562), nhưng trên Test split cả 3 phương thức đều chưa đạt ngưỡng 90%:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6), sharey=True)

methods = ['Keyword', 'Dense', 'Hybrid']
x = np.arange(len(methods))
width = 0.25

for idx, split in enumerate(['DEV', 'TEST']):
    ax = axes[idx]
    sub = df_comp[df_comp['Split'] == split]
    
    h1 = sub['Hit@1 (%)'].values
    h5 = sub['Hit@5 (%)'].values
    mrr = sub['MRR@5'].values * 100
    
    rects1 = ax.bar(x - width, h1, width, label='Hit@1 (%)', color='#2980B9')
    rects2 = ax.bar(x, h5, width, label='Hit@5 (%)', color='#27AE60')
    rects3 = ax.bar(x + width, mrr, width, label='MRR@5 (*100)', color='#F39C12')
    
    ax.axhline(90, color='#C0392B', linestyle='--', linewidth=1.5, label='Ngưỡng Hit@5 (90%)')
    ax.axhline(75, color='#8E44AD', linestyle=':', linewidth=1.5, label='Ngưỡng MRR@5 (0.75)')
    
    title_suffix = ' (GATE 5 PASS)' if split == 'DEV' else ' (GATE 8 NOT MET)'
    ax.set_title(f'{split} SPLIT (60 câu hỏi){title_suffix}', fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(methods, fontweight='bold')
    ax.set_ylabel('Phần trăm (%) / Điểm số')
    ax.set_ylim(0, 115)
    ax.legend(loc='upper left', frameon=True)
    
    for rect in rects1 + rects2 + rects3:
        height = rect.get_height()
        ax.annotate(f'{height:.1f}',
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 3), textcoords='offset points',
                    ha='center', va='bottom', fontsize=9)

plt.suptitle('SO SÁNH HIỆU NĂNG RETRIEVAL: KEYWORD vs DENSE vs HYBRID RRF', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 5. Biểu đồ 2: Đối soát 6 Cổng Chất lượng (Quality Gates Comparison)

Biểu đồ so sánh 6 tiêu chí: Dev Split vượt toàn diện 6/6 tiêu chí; Test Split đạt tiêu chuẩn về An toàn và Hạ tầng (100% Safe Abstention, 0 Scope Leakage), nhưng chưa đạt Retrieval Gate:

In [ ]:
categories = ['Hit@5\n(>=90%)', 'MRR@5\n(>=0.75)', 'Safe Abstention\n(>=95%)', 'Zero Leakage\n(100%)', 'Controlled Safety\n(100%)', 'Infra Stability\n(100%)']

dev_scores = [95.0, 75.62, 100.0, 100.0, 100.0, 100.0]
test_scores = [65.0, 50.62, 100.0, 100.0, 100.0, 100.0]
thresholds = [90.0, 75.0, 95.0, 100.0, 100.0, 100.0]

x = np.arange(len(categories))
width = 0.35

plt.figure(figsize=(13, 6))
plt.bar(x - width/2, dev_scores, width, label='Dev Split (Hybrid RRF — ĐẠT)', color='#2ECC71')
plt.bar(x + width/2, test_scores, width, label='Test Split (Held-out — CHƯA ĐẠT RETRIEVAL)', color='#E67E22')
plt.plot(x, thresholds, color='#E74C3C', marker='o', linewidth=2, linestyle='--', label='Ngưỡng chuẩn quy chế')

plt.xticks(x, categories, fontweight='bold')
plt.ylabel('Điểm số / Tỷ lệ Đạt chuẩn (%)')
plt.ylim(0, 118)
plt.title('ĐỐI SOÁT 6 TIÊU CHÍ CHẤT LƯỢNG RAG BENCHMARK V2', fontsize=15, fontweight='bold')
plt.legend(loc='upper right', frameon=True)

for i in range(len(categories)):
    plt.text(x[i] - width/2, dev_scores[i] + 2, f'{dev_scores[i]:.1f}%', ha='center', fontsize=9, fontweight='bold', color='#1E8449')
    plt.text(x[i] + width/2, test_scores[i] + 2, f'{test_scores[i]:.1f}%', ha='center', fontsize=9, fontweight='bold', color='#B9770E')

plt.tight_layout()
plt.show()

## 6. Biểu đồ 3: Ma trận An toàn & Kiểm soát Từ chối (Safe Abstention Heatmap)

Chứng minh khả năng kiểm soát an toàn: 100% câu hỏi ngoài phạm vi và thiếu chứng cứ (20/20 câu) được từ chối an toàn ở tầng Controlled Retrieval:

In [ ]:
safety_matrix = np.array([
    [26, 14, 0],   # Answerable: 26 Hit@5, 14 Miss, 0 Rò rỉ
    [0, 0, 10],    # Insufficient Evidence: 10 Từ chối an toàn
    [0, 0, 10],    # Scope / Version Security: 10 Chặn thẩm quyền
])

plt.figure(figsize=(9, 6))
plt.imshow(safety_matrix, cmap='Blues', interpolation='nearest')
plt.title('MA TRẬN KIỂM SOÁT AN TOÀN & PHÂN LOẠI CÂU HỎI (TEST SPLIT)', fontsize=13, fontweight='bold', pad=15)

x_labels = ['Truy xuất Chuẩn\n(Hit@5)', 'Phân kỳ Thứ hạng\n(Miss@5)', 'Từ chối An toàn\n(Safe Abstention)']
y_labels = ['Answerable\n(40 câu)', 'Insufficient Evidence\n(10 câu)', 'Scope / Version Security\n(10 câu)']

plt.xticks(range(3), x_labels, fontweight='bold')
plt.yticks(range(3), y_labels, fontweight='bold')
plt.ylabel('Phân loại Câu hỏi', fontweight='bold')
plt.xlabel('Hành vi Ứng xử của RAG Runtime', fontweight='bold')

for i in range(3):
    for j in range(3):
        val = safety_matrix[i, j]
        color = 'white' if val > 15 else 'black'
        plt.text(j, i, str(val), ha='center', va='center', fontsize=16, fontweight='bold', color=color)

plt.tight_layout()
plt.show()

## 7. Biểu đồ 4: Phân tích Chẩn đoán Hậu kiểm 14 Ca Phân kỳ Thứ hạng

> **LƯU Ý:** Đây là chẩn đoán lỗi hậu kiểm (*Diagnostic Error Analysis*) phục vụ cải tiến nghiên cứu, KHÔNG PHẢI là lý do chuyển kết quả từ FAIL thành PASS.

In [ ]:
causes = [
    'Trích xuất đúng Điều khoản Chi tiết\n(Substantive Clause vs Title Page)',
    'Quy chế chung bao hàm Quy chế ngành\n(Cross-regulation Coverage)',
    'Từ khóa đặc thù chuyên sâu\n(Out-of-vocabulary terms)'
]
counts = [11, 2, 1]
colors = ['#27AE60', '#F39C12', '#E74C3C']

plt.figure(figsize=(10, 5))
bars = plt.barh(causes, counts, color=colors)
plt.title('CHẨN ĐOÁN HẬU KIỂM 14 CA PHÂN KỲ THỨ HẠNG TRÊN TẬP TEST', fontsize=14, fontweight='bold')
plt.xlabel('Số lượng câu hỏi (Tổng số: 14 câu)', fontweight='bold')
plt.xlim(0, 14)

for bar in bars:
    w = bar.get_width()
    plt.text(w + 0.3, bar.get_y() + bar.get_height()/2, f'{w} câu ({w/14*100:.1f}%)',
             va='center', fontweight='bold', fontsize=11)

plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 8. Kết luận Khoa học & Kế hoạch Hành động Kế tiếp

1. **Bảo toàn An toàn Tuyệt đối**: Cơ chế Controlled Retrieval fail-closed vận hành hoàn hảo với **100% Safe Abstention** và **0 Scope Leakage**.
2. **Tập held-out Test phản ánh thực tế khách quan**: Đạt Hit@5 = 65.0% và MRR@5 = 0.5062, chưa đạt ngưỡng chất lượng 90%. Điều này chứng minh quy trình kiểm thử hoàn toàn trung thực, không data snooping và không tự sửa kết quả.
3. **Bước đi tiếp theo**: Bổ sung bộ Re-ranker (cross-encoder) và tinh chỉnh prompt template trước khi mở cổng đánh giá sinh câu trả lời bằng Gemini (Gemini Generation Gate).